## Case Study 1: Hospital Readmission Prediction
Logistic Regression (L2 regularized) to predict 30-day readmission risk

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('hospital_readmission_data.csv')

preview first 5 rows

In [ ]:
df.head()

,age,sex,diagnosis_code,n_prior_admissions,n_prior_er_visits,length_of_stay,num_medications,num_diagnoses,systolic_bp,heart_rate,glucose,creatinine,has_diabetes,has_heart_failure,discharge_disposition,readmitted_30d
0,69.6,F,3,0,0,2.8,7,4,103.9,72.9,124.8,0.74,1,0,home,0
1,49.4,M,7,0,0,2.9,5,8,110.1,59.3,163.2,0.76,0,0,home,0
2,76.3,M,4,0,1,4.3,9,3,124.3,115.8,107.6,1.38,0,0,home,0
3,79.1,M,1,0,0,8.2,5,8,101.0,74.5,171.6,0.82,0,0,home,1
4,35.7,M,6,1,1,3.2,15,6,105.5,69.2,94.9,1.39,0,0,snf,0


check no of rows and columns

In [ ]:
df.shape

(5000, 16)

check column data types and non-null counts

In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   age                    5000 non-null   float64
 1   sex                    5000 non-null   str    
 2   diagnosis_code         5000 non-null   int64  
 3   n_prior_admissions     5000 non-null   int64  
 4   n_prior_er_visits      5000 non-null   int64  
 5   length_of_stay         5000 non-null   float64
 6   num_medications        5000 non-null   int64  
 7   num_diagnoses          5000 non-null   int64  
 8   systolic_bp            5000 non-null   float64
 9   heart_rate             5000 non-null   float64
 10  glucose                5000 non-null   float64
 11  creatinine             5000 non-null   float64
 12  has_diabetes           5000 non-null   int64  
 13  has_heart_failure      5000 non-null   int64  
 14  discharge_disposition  5000 non-null   str    
 15  readmitted_30d 

summary of dataset

In [ ]:
df.describe()

,age,diagnosis_code,n_prior_admissions,n_prior_er_visits,length_of_stay,num_medications,num_diagnoses,systolic_bp,heart_rate,glucose,creatinine,has_diabetes,has_heart_failure,readmitted_30d
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,64.650500,4.466400,1.222200,0.801600,4.068080,8.023600,5.989200,129.978020,80.032840,121.095580,1.110930,0.300600,0.207600,0.229600
std,14.846088,2.877521,1.111519,0.898888,2.879135,2.887059,2.237964,20.150961,15.056215,37.598283,0.468919,0.458565,0.405629,0.420618
min,18.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,80.000000,40.000000,60.000000,0.300000,0.000000,0.000000,0.000000
25%,54.600000,2.000000,0.000000,0.000000,1.900000,6.000000,4.000000,115.800000,69.800000,93.300000,0.770000,0.000000,0.000000,0.000000
50%,64.900000,4.000000,1.000000,1.000000,3.300000,8.000000,6.000000,129.900000,80.100000,119.450000,1.100000,0.000000,0.000000,0.000000
75%,74.500000,7.000000,2.000000,1.000000,5.500000,10.000000,7.000000,143.500000,90.300000,146.125000,1.440000,1.000000,0.000000,0.000000
max,100.000000,9.000000,7.000000,6.000000,25.100000,20.000000,14.000000,206.000000,135.700000,265.700000,2.860000,1.000000,1.000000,1.000000


check for missing values in each column

In [ ]:
print(df.isnull().sum())

age                      0
sex                      0
diagnosis_code           0
n_prior_admissions       0
n_prior_er_visits        0
length_of_stay           0
num_medications          0
num_diagnoses            0
systolic_bp              0
heart_rate               0
glucose                  0
creatinine               0
has_diabetes             0
has_heart_failure        0
discharge_disposition    0
readmitted_30d           0
dtype: int64


check for duplicate rows

In [ ]:
print(df.duplicated().sum())

0


Convert text columns (sex, discharge_disposition) into numeric 0/1 columns so the model can use them

In [ ]:
df = pd.get_dummies(df, columns=['sex','discharge_disposition'], drop_first=True)
df.head()

,age,diagnosis_code,n_prior_admissions,n_prior_er_visits,length_of_stay,num_medications,num_diagnoses,systolic_bp,heart_rate,glucose,creatinine,has_diabetes,has_heart_failure,readmitted_30d,sex_M,discharge_disposition_home_health,discharge_disposition_other,discharge_disposition_snf
0,69.6,3,0,0,2.8,7,4,103.9,72.9,124.8,0.74,1,0,0,False,False,False,False
1,49.4,7,0,0,2.9,5,8,110.1,59.3,163.2,0.76,0,0,0,True,False,False,False
2,76.3,4,0,1,4.3,9,3,124.3,115.8,107.6,1.38,0,0,0,True,False,False,False
3,79.1,1,0,0,8.2,5,8,101.0,74.5,171.6,0.82,0,0,1,True,False,False,False
4,35.7,6,1,1,3.2,15,6,105.5,69.2,94.9,1.39,0,0,0,True,False,False,True


Split into features (x) and target (y). Target = readmitted_30d (1 = readmitted within 30 days)

In [ ]:
x = df.drop('readmitted_30d', axis=1)
y = df['readmitted_30d']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

Train Logistic Regression model\nNote: sklearn's LogisticRegression uses L2 regularization by default (penalty='l2', C=1.0)

In [ ]:
log_model = LogisticRegression(max_iter=5000, penalty='l2', C=1.0)
log_model.fit(x_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass`

Predict on test data

In [ ]:
y_pred = log_model.predict(x_test)
y_prob = log_model.predict_proba(x_test)[:,1]   # probability of readmission, needed for ROC-AUC

Evaluate: Accuracy and ROC-AUC

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("Accuracy: ", round(accuracy,4))
print("ROC-AUC: ", round(auc,4))

Accuracy:  0.785
ROC-AUC:  0.6822
